Calculate a matrix-matrix multiplication with OpenCL.

In [ ]:
!pip install pyopencl

In [ ]:
import numpy as np
import pyopencl as cl
import pyopencl.array as cl_array
import time

In [ ]:
platform = cl.get_platforms()[0]
device = platform.get_devices()[0]
print("Platform name:", platform.name)
print("Device name:", device.name)
print("Maximum work group size:", device.max_work_group_size)

In [ ]:
ctx = cl.create_some_context()
queue = cl.CommandQueue(ctx, properties=cl.command_queue_properties.PROFILING_ENABLE)

In the following, we wish to divide the data in smaller workgroups. For this purpose, we will define the workgroup size and the number of data points. Later, the program can only be used when the workgroup size is a divisor of the number of data points. If this is not the case, one needs to pad the data array with dummy variables.

In [ ]:
workgroup_size = 2**5
n_workgroups = 8
matrix_size = workgroup_size * n_workgroups
print("Workgroup size:", workgroup_size)
print("Matrix size:", matrix_size)

In [ ]:
kernel = """
__kernel void matmat(__global const double *inputA,
                     __global const double *inputB,
                     __global double *outputC)
{
  int n_cols = get_global_size(1);
  int global_id_row = get_global_id(0);
  int global_id_col = get_global_id(1);

  double local_sum = 0.0;
  for(int i = 0; i < n_cols; i++){
    local_sum += inputA[global_id_row * n_cols + i] * inputB[global_id_col + i * n_cols];
  }

  outputC[global_id_col + global_id_row * n_cols] = local_sum;
}
"""

In [ ]:
prg = cl.Program(ctx, kernel).build()

In [ ]:
np.random.seed(0)
np_matrix_A = np.random.rand(matrix_size, matrix_size)
np_matrix_B = np.random.rand(matrix_size, matrix_size)

In [ ]:
## Calculate the matmat with Numpy.
start = time.time()

matmat_np = np_matrix_A @ np_matrix_B

time_np = time.time() - start

In [ ]:
## Calculate the matmat with OpenCL.

start = time.time()

cl_matrix_A = cl_array.to_device(queue, np_matrix_A.ravel())
cl_matrix_B = cl_array.to_device(queue, np_matrix_B.ravel())
cl_output = cl_array.empty(queue, matrix_size*matrix_size, dtype=np.float64)

event = prg.matmat(queue,
                   (matrix_size, matrix_size), (workgroup_size, workgroup_size),
                   cl_matrix_A.data, cl_matrix_B.data, cl_output.data)

matmat_cl = cl_output.get().reshape(matrix_size, matrix_size)

time_cl = time.time() - start
time_cl_kernel = 1e-9 * (event.profile.end - event.profile.start)

In [ ]:
print(matmat_np[:5,:5])
print(matmat_cl[:5,:5])

In [ ]:
print("Elapsed time Numpy :", time_np)
print("Elapsed time OpenCL:", time_cl)
print(" of which in kernel:", time_cl_kernel)